# PFN inference on BO contexts: is the predictive calibrated where BO queries, and what does resolution do to the labels?

## Description

<!-- Written 2026-11-12 before the first run (pre-registration). Design: research_log/decisions/0007 §4, tests 1 and 2. Issue #13. -->

**Question.**
- Three pilots found the one-step advantage over EI unpredictable from the frozen PFN's belief: api_probe, api_gate and api_decoder (notebooks 07–09).
- Decision 0007 lists four explanations:
  - E1, irreducible at decision time;
  - E2, PFN approximation error on BO contexts;
  - E3, the PFN's output resolution (1000 bins of 0.001 on the ECDF scale);
  - E4, a label metric that rewards sub-resolution differences.
- This experiment runs the two tests that need no new labels:
  - **test 2** asks whether the PFN does good inference from the data BO collects (E2);
  - **test 1** asks how the labels and the headroom depend on resolution (E3/E4).
- Test 3 (a variance-reduced lookahead, E1 vs E2) and the reference posterior (test 4) are not part of this run.

**Setup.**
- States: the 4000 labeled states of `api_round1/pilot_20261111-133311`.
  - The label set is pinned by sha256, as in notebooks 07 and 08.
  - Split 4, d = 2, n_init = 4, T = 20, one state per task, contexts n = 4 + t ≤ 23.
  - The states come from the round-1 behaviour mix: 80% EI, 10% UCB, 10% PI.
- **Test 2 (E2).** For every state, two contexts of the *same task and the same n*:
  - (a) the BO context (the state's data);
  - (b) a random design: n uniform points from a fixed per-state stream, with their noise-free f.
- Held-out points per state, the same points for both contexts:
  - 16 uniform;
  - 16 local around the BO incumbent (Gaussian, σ ∈ {0.01, 0.05, 0.2}, reflected);
  - EI's pick at the state.
  - The truth is the noise-free f.
- Per point and context:
  - the log density of f(x)'s bin;
  - the randomized PIT (CDF below the bin plus a uniform share of the bin's mass, so an exact predictive gives an exactly uniform PIT);
  - the probabilities of the improvement events {f(x) < incumbent − k bins}, k ∈ {0, 5, 20}, with their outcomes. The incumbent is the context's own.
- *Why this tests calibration validly on BO data.*
  - BO's design and the query points are chosen from the observed data only, so the design is ignorable.
  - If the PFN were the exact posterior under P0, the PIT would be uniform and the event probabilities calibrated over prior draws, for BO and random designs alike. Any deviation on BO contexts is approximation error.
  - The random design is the control. It is near the PFN's training distribution, so it shows whether a deviation is specific to BO contexts.
  - The log scores of the two designs are *not* comparable as a test of E2, because the designs carry different information. The paired log-score difference is reported descriptively only.
  - Strata defined by f\* or by f(x), such as the gap stratum or "resolvable" points, break the calibration guarantee. They are descriptive only.
  - This refines the design in decision 0007 §4, which proposed the paired log-score difference as the main quantity.
- **Test 1 (E3/E4).** By the gap stratum of the BO incumbent, (incumbent − f\*) / 0.001 in PFN bins: < 1, 1–10, 10–100, > 100. Per stratum:
  - the label split-half reliability;
  - the cross-fitted clairvoyant headroom (% of the gap) and each stratum's share of its sum;
  - the share of non-base advantages below one bin in ECDF units;
  - the within-state Spearman of log EI and of PI (at the BO context) with the label.
  - The api_decoder pilot's validation q0.5 score is joined by task seed where its validation states are among these 4000. If there are none, this is skipped and said so.
- Bootstrap: percentile CIs over states, one state per task, 200 resamples in the pilot. BO − random contrasts are paired over states.
- Precision: fp32 (tf32 off). The pilot runs all 4000 states; the sanity run uses 200 states and 50 resamples.

**What to look for.**
1. Test 2, the decision. On BO contexts, for all points pooled and for EI's picks:
   - the 50% and 90% coverage;
   - the KS distance of the PIT;
   - the ECE and calibration-in-the-large (mean predicted minus observed) of each improvement event, against the random design.
2. Test 2, descriptive:
   - PIT histograms by design and point type;
   - reliability diagrams for k = 0 and 5;
   - log density by design, point type and gap stratum.
3. Test 1: the strata table.

**GO / NO-GO.** The rule is `analysis/inference.py::inference_verdict`, on test 2, at all points pooled and at EI's picks.
- **GO (E2 supported)**: on BO contexts, for either point set, one of these holds:
  - a coverage (90% or 50%) is off nominal by ≥ 0.03 with its CI excluding nominal;
  - or some event's ECE exceeds the random design's by ≥ 0.02 with the paired CI > 0.
- **NO-GO (E2 not supported)**: for both point sets, all of these hold:
  - the coverage CIs are within nominal ± 0.03;
  - every event's BO ECE CI is below 0.03;
  - every BO − random ECE CI is below 0.02.
- **INCONCLUSIVE**: otherwise.

Readings, fixed now:
- **GO with the random design calibrated** → E2 at BO contexts: the PFN's inference degrades on the data BO collects.
- **GO with the random design also miscalibrated** → general PFN error, not specific to BO.
- **NO-GO** → the PFN is calibrated on BO data at its resolution, which leaves E1, E3 and E4. Test 1 then says how much of the label variation is below the PFN's resolution.
- **Test 1, E4 signature**:
  - most non-base advantages are below one bin;
  - the strata where the gap is < 1 bin hold most of the states.
  - Neither alone says the headroom is unreal. The headroom in % of the gap may still concentrate where the gap is large.
- **Test 1, E3 limits the PFN where the gap is small** if, in the ≥ 10-bin strata:
  - the within-state Spearman of log EI or PI with the label has a CI lower bound > 0.10;
  - while in the < 1-bin stratum it is ≈ 0, with a CI containing 0.
  - Otherwise, resolution is not what makes the advantage unpredictable.

**Limits.**
- Noise-free P0, d = 2, T = 20 only. pfn_large is valid for n ≤ 100; contexts here have n ≤ 23.
- Calibration tests marginal consistency over prior draws. A calibrated predictive can still be far from the posterior in sharpness, so NO-GO does not prove the PFN equals the posterior. That needs the reference posterior (0007 test 4).
- Exchangeable points within a state are scored together. The bootstrap resamples states, so within-state correlation is accounted for.
- The randomized PIT uses the bar distribution's within-bin uniform density. The noise-free truth inside a bin is not uniform, so near 0, where many f values cluster within one bin, the PIT is uniform only up to the within-bin shape.


In [ ]:
from anytime_acq.analysis import load_snapshot, paired_diff, show_header, summarize

SNAPSHOT = "results/pfn_bo_inference/<sweep_id>"  # set after `just snapshot runs/pfn_bo_inference/<sweep_id>`
snap = load_snapshot(SNAPSHOT)
show_header(snap)  # how it was run + configuration, generated from the snapshot

## Results

In [ ]:
# Aggregates use DONE members only; n_failed / n_started / n_missing say what is not in them yet.
summarize(snap.runs, "<metric>")

## Verdict

The GO / NO-GO criterion from the description, evaluated, in one line. Record it with `just decide`.